# Hey Wony — openWakeWord Custom Model Training

**Before running:** Runtime → Change runtime type → **GPU** (T4 is fine).

Run cells top-to-bottom. Total time: ~2–4 hours on T4 GPU at high-robustness settings.

At the end, download `hey_wony.onnx` and place it at `models/hey_wony.onnx` in the Wony repo,
then set `voice.wake_word.model_path: models/hey_wony.onnx` in your `config.yaml`.

## 1. Install Dependencies

In [ ]:
%%bash
# Install system tools first
apt-get install -y -qq unzip ffmpeg

# Piper TTS sample generator v2.0.0 — main branch removed generate_samples.py
if [ ! -f piper-sample-generator/generate_samples.py ]; then
  rm -rf piper-sample-generator
  git clone --branch v2.0.0 --depth 1 https://github.com/rhasspy/piper-sample-generator
fi
wget -q -O piper-sample-generator/models/en_US-libritts_r-medium.pt \
  'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'
pip install -q piper-phonemize webrtcvad

# openWakeWord + required ONNX feature models
if [ ! -d openwakeword ]; then
  git clone https://github.com/dscripka/openwakeword
fi
pip install -q -e ./openwakeword

OWW_MODELS=openwakeword/openwakeword/resources/models
mkdir -p $OWW_MODELS
for MODEL in melspectrogram.onnx embedding_model.onnx; do
  [ -f "$OWW_MODELS/$MODEL" ] || wget -q --show-progress \
    "https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/$MODEL" \
    -O "$OWW_MODELS/$MODEL"
done

# Training dependencies (no tensorflow — only needed for optional tflite, not onnx)
pip install -q \
  mutagen==1.47.0 \
  torchinfo==1.8.0 \
  torchmetrics==1.2.0 \
  speechbrain==0.5.14 \
  audiomentations==0.33.0 \
  torch-audiomentations==0.11.0 \
  acoustics==0.2.6 \
  'scipy<1.15' \
  pronouncing==0.2.0 \
  'numpy<2' \
  'pyarrow>=12,<14' \
  'datasets==2.14.6' \
  deep-phonemizer==0.0.19 \
  soundfile \
  soxr \
  librosa \
  onnx

echo '✓ Done. NOW: Runtime → Restart session, then continue from cell 2.'

## 2. Download Room Impulse Responses (RIRs)

In [ ]:
import os, datasets, librosa, scipy.io.wavfile, numpy as np

output_dir = "./mit_rirs"
os.makedirs(output_dir, exist_ok=True)

if len(os.listdir(output_dir)) > 0:
    print(f"RIRs already present ({len(os.listdir(output_dir))} files), skipping.")
else:
    # Non-streaming load — downloads and decodes audio locally via soundfile
    ds = datasets.load_dataset(
        "davidscripka/MIT_environmental_impulse_responses",
        split="train"
    )
    for i, row in enumerate(ds):
        arr = row['audio']['array']
        sr  = row['audio']['sampling_rate']
        path = (row['audio'].get('path') or '')
        name = path.split('/')[-1] if path else f'rir_{i:05d}.wav'
        if not name.endswith('.wav'):
            name = f'rir_{i:05d}.wav'
        if sr != 16000:
            arr = librosa.resample(arr, orig_sr=sr, target_sr=16000)
        scipy.io.wavfile.write(
            f"{output_dir}/{name}", 16000,
            (arr * 32767).astype(np.int16)
        )

    print(f"RIRs saved to {output_dir}: {len(os.listdir(output_dir))} files")

## 3. Download Background Noise Data

In [ ]:
%%bash
# Download archives
mkdir -p ./background_noise

if [ ! -f esc50.zip ]; then
  wget -q --show-progress https://github.com/karoldvl/ESC-50/archive/master.zip -O esc50.zip
fi
if [ ! -d ESC-50-master/audio ]; then
  rm -rf ESC-50-master
  unzip -q esc50.zip
fi
if [ ! -f musan.tar.gz ]; then
  wget -q --show-progress http://www.openslr.org/resources/17/musan.tar.gz -O musan.tar.gz
fi
if [ ! -d musan/noise ]; then
  tar -xzf musan.tar.gz
fi
echo "Archives ready."

In [ ]:
import os, subprocess, glob

os.makedirs("background_noise", exist_ok=True)
noise_count = len(os.listdir("background_noise"))
if noise_count >= 100:
    print(f"Background noise already present ({noise_count} files), skipping.")
else:
    workdir = os.path.abspath(".")

    def convert(src, dst):
        if os.path.exists(dst):
            return
        r = subprocess.run(
            ["ffmpeg", "-y", "-i", src, "-ar", "16000", "-ac", "1", dst, "-loglevel", "error"],
            capture_output=True, text=True
        )
        if r.returncode != 0:
            print(f"  skip {os.path.basename(src)}: {r.stderr.strip()[:120]}")

    esc_files = glob.glob(os.path.join(workdir, "ESC-50-master", "audio", "*.wav"))
    print(f"Converting {len(esc_files)} ESC-50 files...")
    for f in esc_files:
        convert(f, os.path.join(workdir, "background_noise", os.path.basename(f)))
    print(f"  ESC-50 done. Total: {len(os.listdir('background_noise'))} files")

    musan_files = (
        glob.glob(os.path.join(workdir, "musan", "noise", "**", "*.wav"), recursive=True) +
        glob.glob(os.path.join(workdir, "musan", "music", "**", "*.wav"), recursive=True)
    )
    print(f"Converting {len(musan_files)} MUSAN files...")
    for f in musan_files:
        convert(f, os.path.join(workdir, "background_noise", "musan_" + os.path.basename(f)))
    print(f"Background noise total: {len(os.listdir('background_noise'))} files")

## 4. Download Pre-computed Features

In [ ]:
%%bash
# ~2000 hours of pre-computed audio features (large file — takes a few minutes)
if [ ! -f openwakeword_features_ACAV100M_2000_hrs_16bit.npy ]; then
  wget -q --show-progress \
    'https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy'
else
  echo "ACAV100M features already present, skipping."
fi

# Validation set for false-positive estimation
if [ ! -f validation_set_features.npy ]; then
  wget -q --show-progress \
    'https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy'
else
  echo "Validation features already present, skipping."
fi

echo "Features downloaded."
ls -lh *.npy

## 5. Configure "Hey Wony" Training (HIGH ROBUSTNESS)

Spelling variants (`hey wony`, `hey wany`, `hey woney`) help Piper cover
different phonetic paths to the target sound /ˈwoʊni/.

In [ ]:
import os, yaml

config = yaml.load(
    open("openwakeword/examples/custom_model.yml").read(),
    yaml.Loader
)

config["target_phrase"]               = ["hey wony", "hey wany", "hey woney"]
config["model_name"]                  = "hey_wony"
config["output_dir"]                  = os.path.abspath("./hey_wony")
config["piper_sample_generator_path"] = os.path.abspath("./piper-sample-generator")

# HIGH ROBUSTNESS settings
config["n_samples"]       = 30000
config["n_samples_val"]   = 2000
config["steps"]           = 50000
config["target_accuracy"] = 0.7
config["target_recall"]   = 0.5

config["rir_paths"]        = [os.path.abspath("./mit_rirs")]
config["background_paths"] = [os.path.abspath("./background_noise")]
config["false_positive_validation_data_path"] = os.path.abspath("validation_set_features.npy")
config["feature_data_files"] = {
    "ACAV100M_sample": os.path.abspath("openwakeword_features_ACAV100M_2000_hrs_16bit.npy")
}

with open("hey_wony.yaml", "w") as f:
    yaml.dump(config, f)

print("Config saved. Key settings:")
print(f"  target_phrase              : {config['target_phrase']}")
print(f"  output_dir                 : {config['output_dir']}")
print(f"  piper_sample_generator_path: {config['piper_sample_generator_path']}")
print(f"  n_samples                  : {config['n_samples']}")
print(f"  steps                      : {config['steps']}")

## 6. Phase 1 — Generate Synthetic Clips

Piper TTS generates positive ("hey wony") and adversarial negative examples.
After this cell, **listen to a few clips** to confirm pronunciation before the long train.

In [ ]:
!python openwakeword/openwakeword/train.py \
    --training_config hey_wony.yaml \
    --generate_clips

In [ ]:
# VERIFY PRONUNCIATION — listen to a few positive clips before committing to full train
import os, glob
from IPython.display import Audio, display

pos_clips = glob.glob("hey_wony/positive_clips/*.wav")[:5]
if not pos_clips:
    pos_clips = glob.glob("**/positive*/**/*.wav", recursive=True)[:5]

print(f"Playing {len(pos_clips)} positive clips — do they sound like 'Hey Wony'?")
for clip in pos_clips:
    print(f"  {clip}")
    display(Audio(clip))

**If pronunciation is wrong:** stop here, adjust `target_phrase` spelling in cell 5, re-run cell 5 and cell 6.

## 7. Phase 2 — Augment Clips with Room Acoustics + Noise

In [ ]:
!python openwakeword/openwakeword/train.py \
    --training_config hey_wony.yaml \
    --augment_clips

## 8. Phase 3 — Train Model (~2–3h on T4)

In [ ]:
%%bash
# || true: train.py exits non-zero after saving .onnx when onnx_tf isn't installed (tflite step).
# The .onnx is saved before that failure — we verify it exists below.
python openwakeword/openwakeword/train.py \
    --training_config hey_wony.yaml \
    --train_model || true

ONNX=$(find hey_wony -name "hey_wony.onnx" 2>/dev/null | head -1)
if [ -z "$ONNX" ]; then
  echo "ERROR: Training failed — hey_wony.onnx not produced."
  exit 1
fi
echo "Model saved: $ONNX"

## 9. Verify Output & Download

In [ ]:
import glob, os

onnx_files = glob.glob("**/*.onnx", recursive=True)
print("ONNX models:", onnx_files)

if onnx_files:
    from openwakeword.model import Model
    m = Model(wakeword_models=[onnx_files[0]], inference_framework="onnx")
    print("Model keys:", list(m.prediction_buffer.keys()))
    print("\nExpected key contains 'hey_wony' — looks good if yes.")
else:
    print("No .onnx found — check training output above.")

In [ ]:
# Download the .onnx file
from google.colab import files
import glob

onnx_files = glob.glob("**/*.onnx", recursive=True)
if onnx_files:
    files.download(onnx_files[0])
    print(f"Downloading: {onnx_files[0]}")
    print("\nNext steps:")
    print("  1. Place hey_wony.onnx at: models/hey_wony.onnx (in Wony repo root)")
    print("  2. In config.yaml set:")
    print("     voice:")
    print("       wake_word:")
    print("         enabled: true")
    print("         model_path: models/hey_wony.onnx")
    print("         threshold: 0.5")
else:
    print("No .onnx found. Check cell 8 output for errors.")